In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt

slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

plt.figure(figsize=(8, 14))
plt.imshow(thumb_np)
plt.axis("off")
plt.title("H&E básica")
plt.show()

Paso 2. Intento de contorno básico

Método A: cv2 por color en HSV

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = thumb_np.copy()

hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
H, S, V = cv2.split(hsv)

# máscara básica de tejido
# tejido = saturación moderada/alta o brillo no tan alto
mask_tissue = ((S > 15) | (V < 220)).astype(np.uint8) * 255

# limpieza morfológica
kernel = np.ones((5, 5), np.uint8)
mask_tissue = cv2.morphologyEx(mask_tissue, cv2.MORPH_OPEN, kernel)
mask_tissue = cv2.morphologyEx(mask_tissue, cv2.MORPH_CLOSE, kernel)

# mayor componente conectado
nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask_tissue, connectivity=8)
largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
mask_main = np.uint8(labels == largest) * 255

plt.figure(figsize=(8, 14))
plt.imshow(mask_main, cmap="gray")
plt.axis("off")
plt.title("Máscara básica del espécimen - método cv2 HSV")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = thumb_np.copy()

# =========================================================
# 1. Máscara inicial combinando:
#    - HSV
#    - distancia al color de fondo
# =========================================================
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
H, S, V = cv2.split(hsv)

# estimar color de fondo con los bordes de la imagen
border_pixels = np.concatenate([
    img[0, :, :],
    img[-1, :, :],
    img[:, 0, :],
    img[:, -1, :]
], axis=0).astype(np.float32)

bg_color = np.median(border_pixels, axis=0)

# distancia de cada píxel al color del fondo
dist = np.linalg.norm(img.astype(np.float32) - bg_color[None, None, :], axis=2)

# máscara básica de tejido
mask_hsv = ((S > 15) | (V < 220)).astype(np.uint8) * 255
mask_dist = (dist > 20).astype(np.uint8) * 255

# combinar ambas
mask_tissue = np.maximum(mask_hsv, mask_dist)

# =========================================================
# 2. Limpieza morfológica
# =========================================================
kernel_open = np.ones((5, 5), np.uint8)
kernel_close = np.ones((9, 9), np.uint8)

mask_tissue = cv2.morphologyEx(mask_tissue, cv2.MORPH_OPEN, kernel_open)
mask_tissue = cv2.morphologyEx(mask_tissue, cv2.MORPH_CLOSE, kernel_close)

# =========================================================
# 3. Quedarse con el mayor componente conectado
# =========================================================
nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask_tissue, connectivity=8)
largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
mask_main = np.uint8(labels == largest) * 255

# =========================================================
# 4. Rellenar huecos internos
# =========================================================
h, w = mask_main.shape
flood = mask_main.copy()
flood_mask = np.zeros((h + 2, w + 2), np.uint8)

cv2.floodFill(flood, flood_mask, (0, 0), 255)
holes = cv2.bitwise_not(flood)
mask_filled = cv2.bitwise_or(mask_main, holes)

# =========================================================
# 5. Suavizar un poco el borde
# =========================================================
kernel_smooth = np.ones((7, 7), np.uint8)
mask_smooth = cv2.morphologyEx(mask_filled, cv2.MORPH_CLOSE, kernel_smooth)
mask_smooth = cv2.morphologyEx(mask_smooth, cv2.MORPH_OPEN, kernel_smooth)

# =========================================================
# 6. Sacar contorno externo
# =========================================================
contours, _ = cv2.findContours(mask_smooth, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
contour = max(contours, key=cv2.contourArea)

# opcional: simplificar un poco el contorno
eps = 0.001 * cv2.arcLength(contour, True)
contour_simple = cv2.approxPolyDP(contour, eps, True)

# =========================================================
# 7. Visualización
# =========================================================
overlay = img.copy()
cv2.drawContours(overlay, [contour], -1, (255, 255, 0), 2)       # amarillo
cv2.drawContours(overlay, [contour_simple], -1, (0, 255, 255), 1) # cian

fig, axes = plt.subplots(1, 4, figsize=(18, 10))

axes[0].imshow(mask_main, cmap="gray")
axes[0].axis("off")
axes[0].set_title("Mayor componente")

axes[1].imshow(mask_filled, cmap="gray")
axes[1].axis("off")
axes[1].set_title("Huecos rellenos")

axes[2].imshow(mask_smooth, cmap="gray")
axes[2].axis("off")
axes[2].set_title("Máscara suavizada")

axes[3].imshow(overlay)
axes[3].axis("off")
axes[3].set_title("Contorno sobre thumbnail")

plt.tight_layout()
plt.show()

Método B: alternativa más robusta con distancia al color de fondo

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = thumb_np.copy().astype(np.float32)

# estimar color del fondo a partir de los bordes
border_pixels = np.concatenate([
    img[0, :, :],
    img[-1, :, :],
    img[:, 0, :],
    img[:, -1, :]
], axis=0)

bg_color = np.median(border_pixels, axis=0)

# distancia euclídea al color de fondo
dist = np.linalg.norm(img - bg_color[None, None, :], axis=2)

mask_dist = (dist > 20).astype(np.uint8) * 255

kernel = np.ones((5, 5), np.uint8)
mask_dist = cv2.morphologyEx(mask_dist, cv2.MORPH_OPEN, kernel)
mask_dist = cv2.morphologyEx(mask_dist, cv2.MORPH_CLOSE, kernel)

nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask_dist, connectivity=8)
largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
mask_main2 = np.uint8(labels == largest) * 255

plt.figure(figsize=(8, 14))
plt.imshow(mask_main2, cmap="gray")
plt.axis("off")
plt.title("Máscara básica del espécimen - distancia al fondo")
plt.show()

Paso 3. Imagen solo del contorno

In [ ]:
contours, _ = cv2.findContours(mask_main2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
contour = max(contours, key=cv2.contourArea)

# visualizar solo contorno
canvas = np.zeros_like(mask_main2)
cv2.drawContours(canvas, [contour], -1, 255, 2)

plt.figure(figsize=(8, 14))
plt.imshow(canvas, cmap="gray")
plt.axis("off")
plt.title("Solo contorno base")
plt.show()

Imagen solo del contorno sobre la H&E:

In [ ]:
overlay = thumb_np.copy()
cv2.drawContours(overlay, [contour], -1, (255, 255, 0), 2)

plt.figure(figsize=(8, 14))
plt.imshow(overlay)
plt.axis("off")
plt.title("Contorno base sobre H&E")
plt.show()

Paso 4. Sacar imágenes de nivel 0 de todo el contorno base

In [ ]:
pts = contour[:, 0, :]   # Nx2 en coordenadas thumbnail

step = 20  # cada 20 puntos del contorno
sampled_pts = pts[::step]

print("Número de puntos muestreados:", len(sampled_pts))

In [ ]:
thumb_h, thumb_w = thumb_np.shape[:2]
full_w, full_h = slide.dimensions

scale_x = full_w / thumb_w
scale_y = full_h / thumb_h

sampled_pts_level0 = np.column_stack([
    sampled_pts[:, 0] * scale_x,
    sampled_pts[:, 1] * scale_y
]).astype(int)

print(sampled_pts_level0[:10])

In [ ]:
tile_size = 1024
tiles = []

for i, (cx, cy) in enumerate(sampled_pts_level0):
    x0 = max(0, cx - tile_size // 2)
    y0 = max(0, cy - tile_size // 2)

    # corregir si se sale del slide
    if x0 + tile_size > full_w:
        x0 = full_w - tile_size
    if y0 + tile_size > full_h:
        y0 = full_h - tile_size

    x0 = int(max(0, x0))
    y0 = int(max(0, y0))

    tile = slide.read_region((x0, y0), 0, (tile_size, tile_size)).convert("RGB")
    tile_np = np.array(tile)

    tiles.append({
        "id": i,
        "x0": x0,
        "y0": y0,
        "img": tile_np
    })

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for ax, t in zip(axes, tiles[:6]):
    ax.imshow(t["img"])
    ax.set_title(f"id={t['id']}\nx0={t['x0']}, y0={t['y0']}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================================================
# 0. Abrir slide y thumbnail
# =========================================================
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

full_w, full_h = slide.dimensions
thumb_h, thumb_w = thumb_np.shape[:2]

# =========================================================
# 1. Máscara base + contorno base
#    (usa aquí el método que mejor te haya funcionado)
# =========================================================
img = thumb_np.copy().astype(np.float32)

border_pixels = np.concatenate([
    img[0, :, :],
    img[-1, :, :],
    img[:, 0, :],
    img[:, -1, :]
], axis=0)

bg_color = np.median(border_pixels, axis=0)
dist = np.linalg.norm(img - bg_color[None, None, :], axis=2)

mask = (dist > 20).astype(np.uint8) * 255

kernel = np.ones((5, 5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
mask_main = np.uint8(labels == largest) * 255

contours, _ = cv2.findContours(mask_main, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
contour = max(contours, key=cv2.contourArea)

# =========================================================
# 2. Muestrear puntos del contorno
# =========================================================
pts = contour[:, 0, :]   # Nx2 en thumbnail

step = 20        # cada cuántos puntos del contorno tomar uno
tile_size = 1024 # tamaño del ROI en level 0

sampled_pts = pts[::step]

# =========================================================
# 3. Convertir puntos a level 0
# =========================================================
scale_x = full_w / thumb_w
scale_y = full_h / thumb_h

sampled_pts_level0 = np.column_stack([
    sampled_pts[:, 0] * scale_x,
    sampled_pts[:, 1] * scale_y
]).astype(int)

# =========================================================
# 4. Construir lista de ROIs level 0
# =========================================================
rois = []

for i, (cx, cy) in enumerate(sampled_pts_level0):
    x0 = max(0, cx - tile_size // 2)
    y0 = max(0, cy - tile_size // 2)

    if x0 + tile_size > full_w:
        x0 = full_w - tile_size
    if y0 + tile_size > full_h:
        y0 = full_h - tile_size

    x0 = int(max(0, x0))
    y0 = int(max(0, y0))

    rois.append({
        "id": i,
        "x0": x0,
        "y0": y0,
        "w": tile_size,
        "h": tile_size,
        "cx": cx,
        "cy": cy
    })

print("Número de ROIs:", len(rois))
print(rois[:5])

# =========================================================
# 5. Dibujar todos los ROIs sobre la H&E global
# =========================================================
fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(thumb_np)

# dibujar contorno base
cv2_contour = contour[:, 0, :]
ax.plot(cv2_contour[:, 0], cv2_contour[:, 1], color="yellow", linewidth=1)

for roi in rois:
    # pasar bbox level 0 a thumbnail
    x_t = roi["x0"] / scale_x
    y_t = roi["y0"] / scale_y
    w_t = roi["w"] / scale_x
    h_t = roi["h"] / scale_y

    rect = Rectangle(
        (x_t, y_t), w_t, h_t,
        fill=False, edgecolor="cyan", linewidth=0.8, alpha=0.8
    )
    ax.add_patch(rect)

ax.set_title("Todos los ROIs de nivel 0 a analizar sobre el contorno base")
ax.axis("off")
plt.show()

NO funciona del todo. Probemos otra cosa

Nueva versión del paso 1: contorno base desde level 4

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================================================
# 0. Abrir slide
# =========================================================
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

print("Level dimensions:", slide.level_dimensions)
print("Level downsamples:", slide.level_downsamples)

# =========================================================
# 1. Elegir imagen base para visualizar: level 4
# =========================================================
display_level = 4
display_size = slide.level_dimensions[display_level]

img_display = slide.read_region((0, 0), display_level, display_size).convert("RGB")
img_display = np.array(img_display)

# =========================================================
# 2. Elegir centro del ROI en coordenadas de level 0
#    (puedes cambiarlo)
# =========================================================
cx0 = 60000
cy0 = 130000

# =========================================================
# 3. Elegir tamaño cuadrado del ROI en píxeles
#    para CADA level
#    (mismo número de píxeles en cada level)
# =========================================================
roi_size_px = 512   # cuadrado de 512x512 en cada level

# =========================================================
# 4. Colores para cada level
# =========================================================
colors = [
    "red", "orange", "yellow", "lime", "cyan",
    "deepskyblue", "blue", "magenta", "white"
]

# =========================================================
# 5. Dibujar ROIs anidados sobre la imagen en display_level
# =========================================================
fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(img_display)

display_downsample = slide.level_downsamples[display_level]

for lvl in range(slide.level_count):
    ds = slide.level_downsamples[lvl]

    # ROI en coordenadas de level 0:
    # si el ROI mide roi_size_px en level=lvl,
    # en level 0 equivale a roi_size_px * ds
    side0 = roi_size_px * ds

    x0 = cx0 - side0 / 2
    y0 = cy0 - side0 / 2

    # convertir a coordenadas de la imagen que mostramos (display_level)
    x_disp = x0 / display_downsample
    y_disp = y0 / display_downsample
    side_disp = side0 / display_downsample

    rect = Rectangle(
        (x_disp, y_disp),
        side_disp,
        side_disp,
        fill=False,
        edgecolor=colors[lvl % len(colors)],
        linewidth=2,
        label=f"level {lvl} ({roi_size_px} px)"
    )
    ax.add_patch(rect)

    # escribir etiqueta cerca de la esquina superior izquierda
    ax.text(
        x_disp,
        y_disp,
        f"L{lvl}",
        color=colors[lvl % len(colors)],
        fontsize=9,
        bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
    )

# marcar centro
ax.plot(cx0 / display_downsample, cy0 / display_downsample, "wo", markersize=4)

ax.set_title(f"ROIs anidados de todos los levels (centro común) sobre level {display_level}")
ax.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================================================
# 0. Abrir slide
# =========================================================
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

print("Level dimensions:", slide.level_dimensions)
print("Level downsamples:", slide.level_downsamples)

# =========================================================
# 1. Elegir imagen base para visualizar: thumbnail
# =========================================================
img_display = slide.associated_images["thumbnail"].convert("RGB")
img_display = np.array(img_display)

thumb_h, thumb_w = img_display.shape[:2]
full_w, full_h = slide.dimensions

# escala desde level 0 -> thumbnail
scale_x = thumb_w / full_w
scale_y = thumb_h / full_h

# =========================================================
# 2. Elegir centro del ROI en coordenadas de level 0
# =========================================================
cx0 = 60000
cy0 = 130000

# =========================================================
# 3. Elegir tamaño cuadrado del ROI en píxeles
#    para CADA level
# =========================================================
roi_size_px = 512   # cuadrado de 512x512 en cada level

# =========================================================
# 4. Colores para cada level
# =========================================================
colors = [
    "red", "orange", "yellow", "lime", "cyan",
    "deepskyblue", "blue", "magenta", "white"
]

# =========================================================
# 5. Dibujar ROIs anidados sobre la thumbnail
# =========================================================
fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(img_display)

for lvl in range(slide.level_count):
    ds = slide.level_downsamples[lvl]

    # si el ROI mide roi_size_px en level=lvl,
    # en level 0 equivale a roi_size_px * ds
    side0 = roi_size_px * ds

    x0 = cx0 - side0 / 2
    y0 = cy0 - side0 / 2

    # convertir coordenadas de level 0 -> thumbnail
    x_disp = x0 * scale_x
    y_disp = y0 * scale_y
    w_disp = side0 * scale_x
    h_disp = side0 * scale_y

    rect = Rectangle(
        (x_disp, y_disp),
        w_disp,
        h_disp,
        fill=False,
        edgecolor=colors[lvl % len(colors)],
        linewidth=2,
        label=f"level {lvl} ({roi_size_px} px)"
    )
    ax.add_patch(rect)

    ax.text(
        x_disp,
        y_disp,
        f"L{lvl}",
        color=colors[lvl % len(colors)],
        fontsize=9,
        bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
    )

# marcar centro
ax.plot(cx0 * scale_x, cy0 * scale_y, "wo", markersize=4)

ax.set_title("ROIs anidados de todos los levels (centro común) sobre thumbnail")
ax.axis("off")
plt.show()

0.1725 µm/px de level 0 del XML

| Level | Dimensiones (px) | Downsample | Resolución efectiva |
| ----- | ---------------: | ---------: | ------------------: |
| 0     |  131928 × 299672 |          1 |        0.1725 µm/px |
| 1     |   65964 × 149836 |          2 |        0.3450 µm/px |
| 2     |    32982 × 74918 |          4 |        0.6900 µm/px |
| 3     |    16491 × 37459 |          8 |        1.3800 µm/px |
| 4     |     8245 × 18729 |         16 |        2.7600 µm/px |
| 5     |      4122 × 9364 |         32 |        5.5200 µm/px |
| 6     |      2061 × 4682 |         64 |       11.0400 µm/px |
| 7     |      1030 × 2341 |        128 |       22.0800 µm/px |
| 8     |       515 × 1170 |        256 |       44.1600 µm/px |


22.76 mm de ancho

51.69 mm de alto

APROXIMADO

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt

slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

plt.figure(figsize=(8, 14))
plt.imshow(thumb_np)
plt.axis("off")
plt.title("H&E básica")
plt.show()

print("thumbnail shape (numpy):", thumb_np.shape)   # (alto, ancho, canales)
print("thumbnail size (PIL):", thumb.size)          # (ancho, alto)

In [ ]:
mpp0 = 0.1725  # micras/pixel en level 0
full_w, full_h = slide.dimensions
thumb_w, thumb_h = thumb.size

# cuánto representa 1 pixel del thumbnail en micras
thumb_mpp_x = mpp0 * (full_w / thumb_w)
thumb_mpp_y = mpp0 * (full_h / thumb_h)

print("thumbnail mpp aprox:", thumb_mpp_x, thumb_mpp_y, "µm/px")
print("tamaño físico aprox del thumbnail:",
      thumb_w * thumb_mpp_x / 1000, "mm x",
      thumb_h * thumb_mpp_y / 1000, "mm")

In [ ]:
print("img_display shape:", img_display.shape)   # (alto, ancho, 3)
print("thumbnail width, height:", thumb_w, thumb_h)
print("full slide width, height:", full_w, full_h)
print("scale_x:", scale_x)
print("scale_y:", scale_y)

en level 0: ROI de 512 x 512 px
en level 1: ROI de 512 x 512 px de ese level, que en level 0 equivale a 1024 x 1024 px
en level 2: equivale a 2048 x 2048 px

In [ ]:
mpp0 = 0.1725  # µm/px en level 0

print("\nROI equivalente de cada level:")
for lvl in range(slide.level_count):
    ds = slide.level_downsamples[lvl]

    # tamaño del ROI expresado en level actual
    side_lvl = roi_size_px

    # tamaño equivalente en level 0
    side0 = roi_size_px * ds

    # tamaño dibujado sobre thumbnail
    w_disp = side0 * scale_x
    h_disp = side0 * scale_y

    # tamaño físico real
    side_um = side0 * mpp0
    side_mm = side_um / 1000

    print(
        f"Level {lvl}: "
        f"ROI = {side_lvl}x{side_lvl} px en ese level | "
        f"equivale a {side0:.0f}x{side0:.0f} px en level 0 | "
        f"en thumbnail = {w_disp:.2f}x{h_disp:.2f} px | "
        f"tamaño físico ≈ {side_mm:.3f} x {side_mm:.3f} mm"
    )

Con roi_size_px = 512 y tus downsamples:

level 0 → 512 px en level 0
level 1 → 1024 px equivalentes en level 0
level 2 → 2048 px
level 3 → 4096 px
level 4 → 8192 px
level 5 → 16384 px
level 6 → 32768 px
level 7 → 65536 px
level 8 → 131072 px

Y en tamaño físico, usando 0.1725 µm/px:

level 0 → 512 * 0.1725 = 88.32 µm = 0.088 mm
level 1 → 0.177 mm
level 2 → 0.353 mm
level 3 → 0.706 mm
level 4 → 1.413 mm
level 5 → 2.826 mm
level 6 → 5.652 mm
level 7 → 11.305 mm
level 8 → 22.610 mm